In [1]:
# ==========================================
# CONTROLE DE THREADS INTERNAS
# ==========================================
# Deve vir antes de importar NumPy / TensorFlow / bibliotecas nativas.
import os

os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("NUMEXPR_NUM_THREADS", "1")
os.environ.setdefault("VECLIB_MAXIMUM_THREADS", "1")

import sys
import time
import numpy as np
import pandas as pd
import multiprocessing
import gc
import shutil
import tempfile
import uuid
from pathlib import Path
import joblib
from scipy.stats import norm
from joblib import Parallel, delayed
import concurrent.futures


# ML e Métricas
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder, LabelEncoder, label_binarize
from sklearn.compose import ColumnTransformer
from sklearn.metrics import (
    accuracy_score, recall_score, f1_score, roc_auc_score,
    precision_score, matthews_corrcoef, precision_recall_curve,
    auc, average_precision_score
)

# Deep Learning (Ataques)
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense, Dropout, Activation
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.utils import to_categorical
import tensorflow as tf
from cleverhans.tf2.attacks.fast_gradient_method import fast_gradient_method
from cleverhans.tf2.attacks.carlini_wagner_l2 import carlini_wagner_l2
from tqdm import tqdm

# Biblioteca oficial mantida importada para compatibilidade, mas esta versão NÃO usa wisardpkg no worker.
import wisardpkg

# ==========================================
# PAINEL DE CONTROLE DO EXPERIMENTO
# ==========================================
DATASETS_TO_RUN = ['Bot-IoT', 'UNSW-NB15', 'CICIDS']

RUN_BINARY = True
RUN_MULTICLASS = True

# Altere conforme necessário:
ENCODING_TYPES = ['linear', 'gaussian', 'distributive']

# ATTACKS_TO_RUN = ['FGSM', 'RANDOM_LINF', 'RANDOM_L2']
ATTACKS_TO_RUN = ['C&W']

EPSILON_LINF = 0.3
EPSILON_L2 = 3.0

# -1 = tenta usar todos os núcleos disponíveis menos 1,
# mas a função de segurança reduz se RAM/disco não forem seguros.
N_JOBS = -1

# ==========================================
# CONTROLE DE PARALELISMO / MEMÓRIA / DISCO
# ==========================================
STANDARD_CHUNKS_PER_WORKER = 4

STANDARD_MEMMAP_ROOT = None

STANDARD_MEMMAP_ROOT_CANDIDATES = [
    r"D:\wisard_memmap",
    r"E:\wisard_memmap",
    r"F:\wisard_memmap",
    r"C:\wisard_memmap",
    str(Path.cwd() / "wisard_memmap"),
]

STANDARD_MIN_FREE_DISK_GB = 80
STANDARD_DISK_SAFETY_MULTIPLIER = 1.35

STANDARD_FORCE_WORKER_SHUTDOWN = True
STANDARD_CLEAN_OLD_MEMMAP_DIRS = True

STANDARD_MEMORY_SAFETY_FRACTION = 0.60
STANDARD_RESERVED_MEMORY_GB = 48

# Nesta versão não há mais .tolist() do treino, mas mantemos uma margem conservadora
# para as tabelas de contagem e estruturas temporárias.
STANDARD_ESTIMATED_CPP_MODEL_GB = 4

WISARD_RANDOM_SEED = 42

# ==========================================
# FAST COUNTING WiSARD COM BLEACHING EXATO
# ==========================================
USE_FAST_COUNTING_WISARD = True

# None = bleaching exato sem teto artificial.
# Exemplo: 200 limita os thresholds de bleaching até 200.
FAST_BLEACH_MAX = None

# Batch interno de predição dentro de cada worker.
# Se a RAM subir muito, reduza para 512.
# Se estiver folgado, pode aumentar para 2048.
FAST_PREDICT_BATCH_SIZE = 2048

# ==========================================
# DIRETÓRIOS DE SAÍDA
# ==========================================
# CSVs pequenos continuam na pasta local "relatorios final".
# Arquivos .npz de curvas podem ser grandes, então vão para o disco grande.
OUTPUT_ROOT = None

OUTPUT_ROOT_CANDIDATES = [
    r"D:\wisard_outputs",
    r"E:\wisard_outputs",
    r"F:\wisard_outputs",
    str(Path.cwd() / "wisard_outputs"),
]

OUTPUT_MIN_FREE_GB = 20

SAVE_CURVES_DATA = True
CURVES_SAVE_COMPRESSED = True


def _existing_parent_for_output(path):
    p = Path(path).expanduser().resolve()

    while not p.exists() and p.parent != p:
        p = p.parent

    return p if p.exists() else None


def _choose_output_root():
    candidates = []

    if OUTPUT_ROOT is not None:
        candidates.append(OUTPUT_ROOT)

    candidates.extend(OUTPUT_ROOT_CANDIDATES)

    checked = []

    for candidate in candidates:
        try:
            parent = _existing_parent_for_output(candidate)

            if parent is None:
                checked.append((candidate, "parent_not_found", 0))
                continue

            free = shutil.disk_usage(parent).free
            checked.append((candidate, "ok", free))

            if free >= OUTPUT_MIN_FREE_GB * (1024 ** 3):
                os.makedirs(candidate, exist_ok=True)
                return Path(candidate).resolve()

        except Exception as e:
            checked.append((candidate, f"error={e}", 0))

    checked_msg = "\n".join(
        f" - {path} | status={status} | livre={free / (1024**3):.1f} GB"
        for path, status, free in checked
    )

    raise OSError(
        "Nenhum diretório de saída tem espaço suficiente.\n"
        f"Espaço mínimo exigido: {OUTPUT_MIN_FREE_GB} GB\n"
        f"Pastas verificadas:\n{checked_msg}"
    )


OUTPUT_BASE_DIR = _choose_output_root()

# CSVs na pasta local do projeto:
REPORTS_DIR = Path("relatorios final").resolve()

# Curvas no disco grande:
CURVES_DIR = OUTPUT_BASE_DIR / "curves_data"

REPORTS_DIR.mkdir(parents=True, exist_ok=True)
CURVES_DIR.mkdir(parents=True, exist_ok=True)


def save_curve_npz(file_path, **kwargs):
    """
    Salva curvas em .npz no diretório configurado em disco grande.
    Usa arquivo temporário + os.replace para evitar arquivo corrompido.
    """
    if not SAVE_CURVES_DATA:
        return False

    file_path = Path(file_path)
    file_path.parent.mkdir(parents=True, exist_ok=True)

    estimated_bytes = 0

    for value in kwargs.values():
        try:
            estimated_bytes += np.asarray(value).nbytes
        except Exception:
            estimated_bytes += 1024

    required = int(estimated_bytes * 1.50) + int(512 * 1024 * 1024)
    free = shutil.disk_usage(file_path.parent).free

    if free < required:
        print(
            f"     [WARN] Curva não salva por falta de espaço: {file_path.name} | "
            f"necessário≈{required / (1024**3):.2f} GB | "
            f"livre≈{free / (1024**3):.2f} GB"
        )
        return False

    tmp_path = file_path.with_suffix(file_path.suffix + ".tmp")

    try:
        with open(tmp_path, "wb") as f:
            if CURVES_SAVE_COMPRESSED:
                np.savez_compressed(f, **kwargs)
            else:
                np.savez(f, **kwargs)

        os.replace(tmp_path, file_path)
        return True

    except Exception:
        try:
            if tmp_path.exists():
                tmp_path.unlink()
        except Exception:
            pass
        raise


print(f"Datasets Selecionados: {DATASETS_TO_RUN}")
print(f"Modos Ativados: Binário={RUN_BINARY} | Multiclasse={RUN_MULTICLASS}")
print(f"Ataques Ativados: {ATTACKS_TO_RUN}")
print(f"Binarizações: {ENCODING_TYPES}")
print("Implementação: FastCounting WiSARD")
print("Bleaching: EXATO e eficiente em Python")
print("wisardpkg no worker: NÃO")
print("Paralelismo: loky + memmap + n_jobs seguro por RAM/disco")
print(f"Memmap candidates: {STANDARD_MEMMAP_ROOT_CANDIDATES}")
print(f"CSV dir: {REPORTS_DIR}")
print(f"Curves dir: {CURVES_DIR}")

Datasets Selecionados: ['Bot-IoT', 'UNSW-NB15', 'CICIDS']
Modos Ativados: Binário=True | Multiclasse=True
Ataques Ativados: ['C&W']
Binarizações: ['linear', 'gaussian', 'distributive']
Implementação: FastCounting WiSARD
Bleaching: EXATO e eficiente em Python
wisardpkg no worker: NÃO
Paralelismo: loky + memmap + n_jobs seguro por RAM/disco
Memmap candidates: ['D:\\wisard_memmap', 'E:\\wisard_memmap', 'F:\\wisard_memmap', 'C:\\wisard_memmap', 'c:\\Users\\Lucas\\Desktop\\Trabalho Mestrado\\wisard_memmap']
CSV dir: C:\Users\Lucas\Desktop\Trabalho Mestrado\relatorios final
Curves dir: D:\wisard_outputs\curves_data


In [24]:
# ==========================================
# FUNÇÕES DE MODELAÇÃO E MÉTRICAS
# ==========================================

def build_and_train_mlp(X, y_cat, num_classes):
    inputs = Input(shape=(X.shape[1],))
    x = Dense(256, activation='relu')(inputs)
    x = Dropout(0.4)(x)
    x = Dense(128, activation='relu')(x)
    x = Dropout(0.4)(x)
    logits = Dense(num_classes, name='logits')(x)
    outputs = Activation('softmax')(logits)

    model = Model(inputs=inputs, outputs=outputs)
    model.compile(
        loss='categorical_crossentropy',
        optimizer=Adam(0.001),
        metrics=['accuracy']
    )
    model.fit(X, y_cat, batch_size=64, epochs=5, verbose=0)
    return model


def process_data_vectorized_sequential(data, resolution, enc_type, custom_thresholds=None, chunk_size=20000):
    n_samples, n_features = data.shape
    result = np.empty((n_samples, n_features * resolution), dtype=np.int8)

    if enc_type == 'linear':
        indices = np.arange(resolution, dtype=np.int8)

    total_chunks = (n_samples + chunk_size - 1) // chunk_size

    for i in range(total_chunks):
        start = i * chunk_size
        end = min((i + 1) * chunk_size, n_samples)
        chunk = data[start:end]

        if enc_type in ['gaussian', 'distributive']:
            bits = (chunk[:, :, None] >= custom_thresholds[None, :, :]).astype(np.int8)

        elif enc_type == 'linear':
            chunk_clipped = np.clip(chunk, 0.0, 1.0)
            limits = (chunk_clipped * resolution).astype(np.int8)
            bits = (limits[:, :, None] > indices[None, None, :]).astype(np.int8)

        result[start:end] = bits.reshape(chunk.shape[0], -1)

    return result


# ==========================================
# CONTROLE DINÂMICO DE MEMÓRIA / DISCO / WORKERS
# ==========================================

def _get_available_memory_bytes():
    try:
        import psutil
        return int(psutil.virtual_memory().available)
    except Exception:
        pass

    if os.name == "nt":
        try:
            import ctypes

            class MEMORYSTATUSEX(ctypes.Structure):
                _fields_ = [
                    ("dwLength", ctypes.c_ulong),
                    ("dwMemoryLoad", ctypes.c_ulong),
                    ("ullTotalPhys", ctypes.c_ulonglong),
                    ("ullAvailPhys", ctypes.c_ulonglong),
                    ("ullTotalPageFile", ctypes.c_ulonglong),
                    ("ullAvailPageFile", ctypes.c_ulonglong),
                    ("ullTotalVirtual", ctypes.c_ulonglong),
                    ("ullAvailVirtual", ctypes.c_ulonglong),
                    ("sullAvailExtendedVirtual", ctypes.c_ulonglong),
                ]

            stat = MEMORYSTATUSEX()
            stat.dwLength = ctypes.sizeof(MEMORYSTATUSEX)
            ctypes.windll.kernel32.GlobalMemoryStatusEx(ctypes.byref(stat))
            return int(stat.ullAvailPhys)

        except Exception:
            return None

    try:
        pages = os.sysconf("SC_AVPHYS_PAGES")
        page_size = os.sysconf("SC_PAGE_SIZE")
        return int(pages * page_size)
    except Exception:
        return None


def _existing_parent(path):
    p = Path(path).expanduser().resolve()

    while not p.exists() and p.parent != p:
        p = p.parent

    return p if p.exists() else None


def _estimate_memmap_required_bytes(X_train_bin, y_train_curr, X_test_bin, active_adv_dict):
    total = 0

    total += int(np.ascontiguousarray(X_train_bin).nbytes)
    total += int(np.asarray(y_train_curr, dtype=np.int32).nbytes)
    total += int(np.ascontiguousarray(X_test_bin).nbytes)

    for X_adv in active_adv_dict.values():
        total += int(np.ascontiguousarray(X_adv).nbytes)

    total = int(total * float(STANDARD_DISK_SAFETY_MULTIPLIER))
    total += int(STANDARD_MIN_FREE_DISK_GB * (1024 ** 3))

    return total


def _cleanup_standard_memmap_leftovers(root):
    if not STANDARD_CLEAN_OLD_MEMMAP_DIRS:
        return

    root = Path(root)

    if not root.exists():
        return

    for p in root.glob("standard_wisard_memmap_*"):
        try:
            shutil.rmtree(p, ignore_errors=True)
        except Exception:
            pass


def _get_standard_memmap_root(required_bytes=0):
    candidates = []

    if STANDARD_MEMMAP_ROOT is not None:
        candidates.append(STANDARD_MEMMAP_ROOT)

    candidates.extend(STANDARD_MEMMAP_ROOT_CANDIDATES)
    candidates.append(tempfile.gettempdir())

    checked = []

    for candidate in candidates:
        try:
            parent = _existing_parent(candidate)

            if parent is None:
                checked.append((candidate, "parent_not_found", 0))
                continue

            usage = shutil.disk_usage(parent)
            free = int(usage.free)

            checked.append((candidate, "ok", free))

            if free >= int(required_bytes):
                os.makedirs(candidate, exist_ok=True)
                _cleanup_standard_memmap_leftovers(candidate)
                return str(Path(candidate).resolve())

        except Exception as e:
            checked.append((candidate, f"error={e}", 0))

    checked_msg = "\n".join(
        f" - {path} | status={status} | livre={free / (1024**3):.1f} GB"
        for path, status, free in checked
    )

    raise OSError(
        "Nenhum diretório de memmap tem espaço livre suficiente.\n"
        f"Espaço estimado necessário, com margem: {required_bytes / (1024**3):.1f} GB\n"
        "Pastas verificadas:\n"
        f"{checked_msg}\n\n"
        "Solução prática: libere espaço em disco ou defina STANDARD_MEMMAP_ROOT "
        "para uma unidade grande, por exemplo r'D:\\wisard_memmap'."
    )


def _estimate_worker_peak_bytes_fast_counting(
    X_train_bin,
    y_train_curr,
    X_test_bin,
    active_adv_dict,
    addr,
    n_jobs_guess
):
    """
    Estimativa aproximada para a FastCountingWiSARD.

    Diferente da wisardpkg, esta versão NÃO converte o treino inteiro para .tolist().
    O pico vem de:
    - vetor de endereços do treino para uma RAM;
    - tabelas de contadores ordenadas;
    - tensor temporário de contagens durante a predição;
    - overhead conservador.
    """
    n_train = int(X_train_bin.shape[0])
    n_bits = int(X_train_bin.shape[1])
    n_test = int(X_test_bin.shape[0])
    addr = int(addr)

    n_rams = int(np.ceil(n_bits / addr))
    n_classes = int(np.max(y_train_curr)) + 1

    address_vec_bytes = n_train * 8

    max_addresses = min(2 ** min(addr, 30), max(1, n_train))
    avg_class_samples = max(1, int(np.ceil(n_train / max(1, n_classes))))
    unique_per_class_ram = min(max_addresses, avg_class_samples)

    # keys uint64 + counts uint32 + overhead aproximado.
    table_bytes = n_rams * n_classes * unique_per_class_ram * 20

    chunks_per_worker = int(STANDARD_CHUNKS_PER_WORKER)
    approx_tasks = max(1, int(n_jobs_guess) * chunks_per_worker)
    approx_chunk_rows = max(1, int(np.ceil(n_test / approx_tasks)))
    pred_batch_rows = min(int(FAST_PREDICT_BATCH_SIZE), approx_chunk_rows)

    pred_tensor_bytes = pred_batch_rows * n_classes * n_rams * 4

    overhead = int(STANDARD_ESTIMATED_CPP_MODEL_GB * (1024 ** 3))

    return int(address_vec_bytes + table_bytes + pred_tensor_bytes * 2 + overhead)


def _resolve_cpu_jobs(n_jobs, n_samples):
    cpu_count = multiprocessing.cpu_count()

    if n_jobs is None or n_jobs == -1:
        resolved = max(1, cpu_count - 1)
    elif n_jobs < 0:
        resolved = max(1, cpu_count + 1 + int(n_jobs))
    else:
        resolved = int(n_jobs)

    return max(1, min(resolved, int(n_samples)))


def _resolve_n_jobs_memory_safe(
    n_jobs,
    n_samples,
    X_train_bin,
    y_train_curr,
    X_test_bin,
    active_adv_dict,
    addr
):
    cpu_jobs = _resolve_cpu_jobs(n_jobs, n_samples)
    available = _get_available_memory_bytes()

    if available is None:
        safe_jobs = min(cpu_jobs, 2)
        print(
            f"     [MEM] Não consegui medir RAM disponível. "
            f"Usando fallback seguro: n_jobs={safe_jobs}/{cpu_jobs}"
        )
        return max(1, safe_jobs)

    reserved = int(STANDARD_RESERVED_MEMORY_GB * (1024 ** 3))
    usable = int(max(1, available * STANDARD_MEMORY_SAFETY_FRACTION - reserved))

    per_worker = _estimate_worker_peak_bytes_fast_counting(
        X_train_bin=X_train_bin,
        y_train_curr=y_train_curr,
        X_test_bin=X_test_bin,
        active_adv_dict=active_adv_dict,
        addr=addr,
        n_jobs_guess=cpu_jobs
    )

    max_jobs_by_memory = max(1, int(usable // max(1, per_worker)))
    resolved = max(1, min(cpu_jobs, max_jobs_by_memory, int(n_samples)))

    print(
        "     [MEM] RAM disponível≈"
        f"{available / (1024**3):.1f} GB | "
        f"RAM utilizável≈{usable / (1024**3):.1f} GB | "
        f"pico estimado/worker≈{per_worker / (1024**3):.1f} GB | "
        f"n_jobs seguro={resolved}/{cpu_jobs}"
    )

    return resolved


def _build_ranges(n_samples, n_jobs, chunks_per_worker=4):
    n_tasks = max(1, min(int(n_samples), int(n_jobs) * int(chunks_per_worker)))
    cuts = np.linspace(0, int(n_samples), n_tasks + 1, dtype=np.int64)

    return [
        (int(cuts[i]), int(cuts[i + 1]))
        for i in range(n_tasks)
        if cuts[i] < cuts[i + 1]
    ]


def _dump_memmap_array(array, folder, name):
    path = os.path.join(folder, f'{name}.joblib')
    arr = np.ascontiguousarray(array)

    required = int(arr.nbytes * 1.10) + int(2 * (1024 ** 3))
    free = shutil.disk_usage(folder).free

    if free < required:
        raise OSError(
            f"Espaço insuficiente ao gravar {name}.joblib.\n"
            f"Necessário≈{required / (1024**3):.1f} GB | "
            f"Livre≈{free / (1024**3):.1f} GB | "
            f"Pasta={folder}"
        )

    joblib.dump(arr, path, compress=0)
    return path


def _safe_rmtree(path, retries=8, sleep_s=1.0):
    path = Path(path)

    if not path.exists():
        return True

    for attempt in range(1, retries + 1):
        try:
            gc.collect()
            shutil.rmtree(path, ignore_errors=False)
            return True

        except Exception as e:
            if attempt == retries:
                print(
                    f"     [WARN] Não consegui apagar temp_dir após {retries} tentativas: {path}\n"
                    f"            Motivo: {repr(e)}\n"
                    f"            Você pode apagar manualmente depois."
                )
                return False

            time.sleep(sleep_s)

    return False


def _shutdown_loky_workers():
    try:
        from joblib.externals.loky import get_reusable_executor
        get_reusable_executor().shutdown(wait=True, kill_workers=True)
    except Exception:
        pass


# --------------------------------------------------------------------------
# MÓDULO IMPORTÁVEL PARA OS WORKERS LOKY
# --------------------------------------------------------------------------

STANDARD_WORKER_MODULE_CODE = r"""
import os

os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("NUMEXPR_NUM_THREADS", "1")
os.environ.setdefault("VECLIB_MAXIMUM_THREADS", "1")

import sys
import time
import random
import gc
import numpy as np
import joblib


_STATE_KEY = None
_MODEL = None
_X_TEST = None
_ADV = None
_TRAIN_TIME = 0.0


class FastCountingWisard:

    # Standard WiSARD exata com counting RAMs e bleaching eficiente.

    # Memória:
    #     table[ram][class] = (keys_sorted, counts)

    # Treino:
    #     count[class][ram][address] += 1

    # Inferência:
    #     score_class(b) = soma(count >= b para todas as RAMs)

    # Bleaching:
    #     em vez de testar b = 1, 2, 3, ..., max_count,
    #     testa apenas thresholds onde algum score pode mudar:
    #         b = count + 1


    def __init__(
        self,
        address_size,
        class_order,
        seed=42,
        bleach_max=None,
        predict_batch_size=1024
    ):
        self.address_size = int(address_size)
        self.class_order = [str(c) for c in class_order]
        self.class_values = np.array([int(c) for c in self.class_order], dtype=np.int32)
        self.n_classes = len(self.class_order)

        self.seed = int(seed)
        self.bleach_max = bleach_max
        self.predict_batch_size = int(predict_batch_size)

        self.mapping = None
        self.powers = np.left_shift(
            np.uint64(1),
            np.arange(self.address_size, dtype=np.uint64)
        ).astype(np.uint64)

        self.n_bits = None
        self.n_rams = None
        self.tables = None

    def _build_mapping(self, n_bits):
        self.n_bits = int(n_bits)
        self.n_rams = int(np.ceil(self.n_bits / self.address_size))

        rng = np.random.default_rng(self.seed)
        indices = np.arange(self.n_bits, dtype=np.int64)
        rng.shuffle(indices)

        total_needed = self.n_rams * self.address_size
        pad = total_needed - self.n_bits

        if pad > 0:
            extra = indices[:pad]
            indices = np.concatenate([indices, extra])

        self.mapping = indices.reshape(self.n_rams, self.address_size)

    def _addresses_for_ram(self, X, ram_idx):
        idx = self.mapping[int(ram_idx)]
        bits = np.asarray(X[:, idx], dtype=np.uint64)
        return bits @ self.powers

    def train(self, X_train, y_train):
        y_train = np.asarray(y_train, dtype=np.int32)

        if self.mapping is None:
            self._build_mapping(X_train.shape[1])

        masks = [(y_train == cls_value) for cls_value in self.class_values]

        tables = []

        for ram_idx in range(self.n_rams):
            addrs = self._addresses_for_ram(X_train, ram_idx)
            ram_tables = []

            for class_idx in range(self.n_classes):
                class_addrs = addrs[masks[class_idx]]

                if class_addrs.size == 0:
                    keys = np.empty(0, dtype=np.uint64)
                    counts = np.empty(0, dtype=np.uint32)
                else:
                    keys, counts = np.unique(class_addrs, return_counts=True)
                    keys = keys.astype(np.uint64, copy=False)
                    counts = counts.astype(np.uint32, copy=False)

                ram_tables.append((keys, counts))

            tables.append(ram_tables)

            del addrs

            if ram_idx % 32 == 0:
                gc.collect()

        self.tables = tables
        gc.collect()

    def _lookup_counts_for_ram_class(self, keys, counts, addresses):
        if keys.size == 0:
            return np.zeros(addresses.shape[0], dtype=np.uint32)

        pos = np.searchsorted(keys, addresses)
        pos_clip = np.minimum(pos, keys.size - 1)

        valid = (pos < keys.size) & (keys[pos_clip] == addresses)

        out = np.zeros(addresses.shape[0], dtype=np.uint32)
        out[valid] = counts[pos_clip[valid]]

        return out

    def _count_tensor_for_batch(self, X_batch):
        n = int(X_batch.shape[0])

        count_tensor = np.zeros(
            (n, self.n_classes, self.n_rams),
            dtype=np.uint32
        )

        for ram_idx in range(self.n_rams):
            addresses = self._addresses_for_ram(X_batch, ram_idx)

            for class_idx in range(self.n_classes):
                keys, counts = self.tables[ram_idx][class_idx]
                count_tensor[:, class_idx, ram_idx] = self._lookup_counts_for_ram_class(
                    keys,
                    counts,
                    addresses
                )

            del addresses

        return count_tensor

    def _first_by_class_order(self, class_indices):
        class_indices = [int(i) for i in class_indices]
        return min(class_indices)

    def _predict_one_from_counts(self, counts_one):
        scores = np.count_nonzero(counts_one >= 1, axis=1)
        max_score = int(np.max(scores))

        tied = np.flatnonzero(scores == max_score)

        if tied.size == 1:
            return int(tied[0])

        tied_counts = counts_one[tied, :]

        candidate_thresholds = np.unique(
            tied_counts[tied_counts > 0].astype(np.uint64) + np.uint64(1)
        )

        candidate_thresholds = candidate_thresholds[candidate_thresholds >= 2]

        if self.bleach_max is not None:
            candidate_thresholds = candidate_thresholds[
                candidate_thresholds <= int(self.bleach_max)
            ]

        if candidate_thresholds.size == 0:
            return int(self._first_by_class_order(tied))

        for b in candidate_thresholds:
            scores_b = np.count_nonzero(tied_counts >= b, axis=1)
            max_score_b = int(np.max(scores_b))

            tied_b_local = np.flatnonzero(scores_b == max_score_b)

            if tied_b_local.size == 1:
                return int(tied[int(tied_b_local[0])])

        return int(self._first_by_class_order(tied))

    def _predict_batch(self, X_batch):
        count_tensor = self._count_tensor_for_batch(X_batch)

        preds = []

        for i in range(count_tensor.shape[0]):
            pred_idx = self._predict_one_from_counts(count_tensor[i])
            preds.append(self.class_order[pred_idx])

        del count_tensor
        gc.collect()

        return preds

    def predict(self, X):
        preds = []
        n = int(X.shape[0])
        batch_size = max(1, int(self.predict_batch_size))

        for start in range(0, n, batch_size):
            end = min(start + batch_size, n)
            preds.extend(self._predict_batch(X[start:end]))

        return preds


def _ensure_state(config):
    global _STATE_KEY, _MODEL, _X_TEST, _ADV, _TRAIN_TIME

    if _STATE_KEY == config['state_key'] and _MODEL is not None:
        return 0.0

    np.random.seed(int(config.get('seed', 42)))
    random.seed(int(config.get('seed', 42)))

    X_train = joblib.load(config['X_train_path'], mmap_mode='r')
    y_train = joblib.load(config['y_train_path'], mmap_mode='r')
    _X_TEST = joblib.load(config['X_test_path'], mmap_mode='r')

    _ADV = {
        name: joblib.load(path, mmap_mode='r')
        for name, path in config['adv_paths'].items()
    }

    t0 = time.perf_counter()

    model = FastCountingWisard(
        address_size=int(config['addr']),
        class_order=config['class_order'],
        seed=int(config.get('seed', 42)),
        bleach_max=config.get('bleach_max', None),
        predict_batch_size=int(config.get('predict_batch_size', 1024))
    )

    model.train(X_train, y_train)

    _TRAIN_TIME = time.perf_counter() - t0

    _MODEL = model
    _STATE_KEY = config['state_key']

    del X_train, y_train
    gc.collect()

    return _TRAIN_TIME


def evaluate_standard_range_worker(config, start, end):
    train_time = _ensure_state(config)

    start = int(start)
    end = int(end)
    pid = os.getpid()

    t1 = time.perf_counter()
    yp_clean = _MODEL.predict(_X_TEST[start:end])
    infer_time_clean = time.perf_counter() - t1

    yp_adv = {}
    infer_time_adv = {}

    for atk_name, adv_matrix in _ADV.items():
        t2 = time.perf_counter()
        yp_adv[atk_name] = _MODEL.predict(adv_matrix[start:end])
        infer_time_adv[atk_name] = time.perf_counter() - t2

    return {
        'pid': pid,
        'start': start,
        'end': end,
        'yp_clean': yp_clean,
        'yp_adv': yp_adv,
        'train_time': float(train_time),
        'infer_time_clean': float(infer_time_clean),
        'infer_time_adv': infer_time_adv,
        'n_rams': int(_MODEL.n_rams),
        'address_size': int(_MODEL.address_size),
        'backend': 'FastCountingWisard',
        'fast_exact_bleaching': True,
    }
"""


def _write_standard_worker_runtime_module(module_path='standard_worker_runtime.py'):
    module_path = Path(module_path)
    module_path.write_text(STANDARD_WORKER_MODULE_CODE, encoding='utf-8')

    import importlib
    importlib.invalidate_caches()

    if str(module_path.parent.resolve()) not in sys.path:
        sys.path.insert(0, str(module_path.parent.resolve()))

    return module_path


def evaluate_standard_persistent_parallel(
    addr,
    X_train_bin,
    y_train_curr,
    X_test_bin,
    active_adv_dict,
    n_jobs=-1,
    chunks_per_worker=None,
):
    """
    Avaliação paralela persistente da Standard WiSARD.

    Esta versão usa FastCountingWisard:
    - counting RAMs exatas;
    - bleaching eficiente;
    - sem wisardpkg.classify;
    - sem .tolist() no treino;
    - memmap + loky mantidos.
    """
    if chunks_per_worker is None:
        chunks_per_worker = STANDARD_CHUNKS_PER_WORKER

    n_samples = len(X_test_bin)

    required_disk = _estimate_memmap_required_bytes(
        X_train_bin=X_train_bin,
        y_train_curr=y_train_curr,
        X_test_bin=X_test_bin,
        active_adv_dict=active_adv_dict
    )

    memmap_root = _get_standard_memmap_root(required_bytes=required_disk)
    os.makedirs(memmap_root, exist_ok=True)

    print(
        f"     [DISK] Memmap root: {memmap_root} | "
        f"livre≈{shutil.disk_usage(memmap_root).free / (1024**3):.1f} GB | "
        f"estimado necessário≈{required_disk / (1024**3):.1f} GB"
    )

    n_jobs_resolved = _resolve_n_jobs_memory_safe(
        n_jobs=n_jobs,
        n_samples=n_samples,
        X_train_bin=X_train_bin,
        y_train_curr=y_train_curr,
        X_test_bin=X_test_bin,
        active_adv_dict=active_adv_dict,
        addr=addr
    )

    ranges = _build_ranges(n_samples, n_jobs_resolved, chunks_per_worker)

    _write_standard_worker_runtime_module()

    import importlib
    runtime = importlib.import_module('standard_worker_runtime')
    runtime = importlib.reload(runtime)

    temp_dir = tempfile.mkdtemp(prefix='standard_wisard_memmap_', dir=memmap_root)
    t_wall = time.perf_counter()

    try:
        X_train_path = _dump_memmap_array(X_train_bin, temp_dir, 'X_train_bin')

        y_train_path = _dump_memmap_array(
            np.asarray(y_train_curr, dtype=np.int32),
            temp_dir,
            'y_train_curr'
        )

        X_test_path = _dump_memmap_array(X_test_bin, temp_dir, 'X_test_bin')

        adv_paths = {
            atk_name: _dump_memmap_array(X_adv, temp_dir, f'adv_{atk_name}')
            for atk_name, X_adv in active_adv_dict.items()
        }

        max_label = int(np.max(y_train_curr))
        class_order = [str(i) for i in range(max_label + 1)]

        config = {
            'state_key': str(uuid.uuid4()),
            'seed': int(WISARD_RANDOM_SEED),
            'addr': int(addr),
            'X_train_path': X_train_path,
            'y_train_path': y_train_path,
            'X_test_path': X_test_path,
            'adv_paths': adv_paths,
            'class_order': class_order,
            'bleach_max': FAST_BLEACH_MAX,
            'predict_batch_size': int(FAST_PREDICT_BATCH_SIZE),
        }

        results = Parallel(
            n_jobs=n_jobs_resolved,
            backend='loky',
            batch_size=1,
            pre_dispatch=n_jobs_resolved,
            max_nbytes=None,
            temp_folder=temp_dir,
            verbose=0,
        )(
            delayed(runtime.evaluate_standard_range_worker)(config, start, end)
            for start, end in ranges
        )

        wall_time = time.perf_counter() - t_wall

    finally:
        if STANDARD_FORCE_WORKER_SHUTDOWN:
            _shutdown_loky_workers()

        gc.collect()
        _safe_rmtree(temp_dir)

    return {
        'results': results,
        'n_jobs': n_jobs_resolved,
        'ranges': ranges,
        'parallel_wall_time': wall_time,
    }


def calculate_miss_rates(y_true, y_pred, y_prob, context_name, class_names):
    metrics = {}

    try:
        normal_idx = next(i for i, name in enumerate(class_names) if 'normal' in str(name).lower())
    except StopIteration:
        normal_idx = 0

    is_binary = len(class_names) == 2
    avg_type = 'binary' if is_binary else 'weighted'
    pos_label = 1 if is_binary else None

    metrics[f'{context_name}_Acc'] = accuracy_score(y_true, y_pred)

    metrics[f'{context_name}_Precision'] = precision_score(
        y_true,
        y_pred,
        average=avg_type,
        pos_label=pos_label,
        zero_division=0
    )

    metrics[f'{context_name}_Recall'] = recall_score(
        y_true,
        y_pred,
        average=avg_type,
        pos_label=pos_label,
        zero_division=0
    )

    metrics[f'{context_name}_F1'] = f1_score(
        y_true,
        y_pred,
        average=avg_type,
        pos_label=pos_label,
        zero_division=0
    )

    metrics[f'{context_name}_MCC'] = matthews_corrcoef(y_true, y_pred)

    mask_normal = (y_true == normal_idx)
    mask_attack = (y_true != normal_idx)

    if np.sum(mask_normal) > 0:
        metrics[f'{context_name}_FAR'] = np.sum((y_pred != normal_idx) & mask_normal) / np.sum(mask_normal)
    else:
        metrics[f'{context_name}_FAR'] = 0.0

    if np.sum(mask_attack) > 0:
        metrics[f'{context_name}_ASR'] = np.sum((y_pred == normal_idx) & mask_attack) / np.sum(mask_attack)
    else:
        metrics[f'{context_name}_ASR'] = 0.0

    try:
        if is_binary:
            prob_positive = y_prob[:, 1] if len(y_prob.shape) > 1 else y_prob
            metrics[f'{context_name}_AUC'] = roc_auc_score(y_true, prob_positive)

            prec, rec, _ = precision_recall_curve(y_true, prob_positive)
            metrics[f'{context_name}_PR_AUC'] = auc(rec, prec)

        else:
            metrics[f'{context_name}_AUC'] = roc_auc_score(y_true, y_prob, multi_class='ovr')
            y_true_bin = label_binarize(y_true, classes=range(len(class_names)))

            metrics[f'{context_name}_PR_AUC'] = average_precision_score(
                y_true_bin,
                y_prob,
                average="macro"
            )

    except Exception:
        metrics[f'{context_name}_AUC'] = 0.0
        metrics[f'{context_name}_PR_AUC'] = 0.0

    if not is_binary:
        metrics[f'{context_name}_F1_Macro'] = f1_score(
            y_true,
            y_pred,
            average='macro',
            zero_division=0
        )

    for idx, name in enumerate(class_names):
        if idx == normal_idx:
            continue

        mask_t = (y_true == idx)

        if np.sum(mask_t) > 0:
            metrics[f'{context_name}_Miss_{name}'] = np.sum(
                (y_pred == normal_idx) & mask_t
            ) / np.sum(mask_t)
        else:
            metrics[f'{context_name}_Miss_{name}'] = 0.0

    return metrics


def append_row_to_csv_safely(row_df, csv_name):
    """
    Mantém append sem quebrar CSV antigo.

    Se o CSV já existe, usa exatamente as colunas do cabeçalho antigo.
    Isso evita corromper arquivos anteriores quando adicionamos colunas novas.
    """
    csv_name = Path(csv_name)
    csv_name.parent.mkdir(parents=True, exist_ok=True)

    if csv_name.exists():
        existing_cols = list(pd.read_csv(csv_name, nrows=0).columns)

        for col in existing_cols:
            if col not in row_df.columns:
                row_df[col] = np.nan

        row_df = row_df[existing_cols]
        row_df.to_csv(csv_name, mode='a', header=False, index=False)

    else:
        row_df.to_csv(csv_name, mode='w', header=True, index=False)

In [25]:
# ==========================================
# LOOP PRINCIPAL DO EXPERIMENTO (FAST COUNTING STANDARD WISARD)
# ==========================================

for dataset_name in DATASETS_TO_RUN:
    print(f"\n{'='*50}")
    print(f">>> A INICIAR EXPERIMENTOS: {dataset_name.upper()}")
    print(f"{'='*50}")

    # 1. CARREGAMENTO E PRÉ-PROCESSAMENTO
    if dataset_name == 'Bot-IoT':
        df_train = pd.read_csv("data2/BotIoT_training-set.csv")
        df_test = pd.read_csv("data2/BotIoT_testing-set.csv")

    elif dataset_name == 'UNSW-NB15':
        df_train = pd.read_csv("data/UNSW_NB15_training-set.csv")
        df_test = pd.read_csv("data/UNSW_NB15_testing-set.csv")

    elif dataset_name == 'CICIDS':
        df_train = pd.read_csv("data3/CICIDS_training-set.csv")
        df_test = pd.read_csv("data3/CICIDS_testing-set.csv")

    else:
        raise ValueError(f"Dataset desconhecido: {dataset_name}")

    for df in [df_train, df_test]:
        if 'id' in df.columns:
            df.drop(columns=['id'], inplace=True)

    y_train_bin = df_train['label'].values
    y_test_bin = df_test['label'].values
    class_names_bin = ['Normal', 'Attack']

    df_train['attack_cat'] = df_train['attack_cat'].astype(str).str.strip().str.lower()
    df_test['attack_cat'] = df_test['attack_cat'].astype(str).str.strip().str.lower()

    le = LabelEncoder()
    le.fit(pd.concat([df_train['attack_cat'], df_test['attack_cat']]))

    y_train_multi = le.transform(df_train['attack_cat'])
    y_test_multi = le.transform(df_test['attack_cat'])
    class_names_multi = le.classes_

    df_train.drop(columns=['label', 'attack_cat'], inplace=True, errors='ignore')
    df_test.drop(columns=['label', 'attack_cat'], inplace=True, errors='ignore')

    categorical_cols = df_train.select_dtypes(include=['object']).columns
    numerical_cols = df_train.select_dtypes(include=['int64', 'float64']).columns

    preprocessor = ColumnTransformer([
        ('num', MinMaxScaler(feature_range=(0, 1)), numerical_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_cols)
    ])

    print(">>> A aplicar Scaling e One-Hot Encoding...")
    preprocessor.fit(df_train)

    X_train = preprocessor.transform(df_train).astype('float32')
    X_test = preprocessor.transform(df_test).astype('float32')

    del df_train, df_test
    gc.collect()

    # 2. GERAÇÃO DOS ATAQUES ADVERSARIAIS E RUÍDOS
    attacks_dict_bin = {}
    attacks_dict_multi = {}

    if 'FGSM' in ATTACKS_TO_RUN:
        print(">>> A gerar Ataque FGSM...")

        if RUN_BINARY:
            mlp_bin = build_and_train_mlp(
                X_train,
                to_categorical(y_train_bin, 2),
                2
            )

            logits_bin = Model(
                inputs=mlp_bin.input,
                outputs=mlp_bin.get_layer('logits').output
            )

            attacks_dict_bin['FGSM'] = fast_gradient_method(
                logits_bin,
                tf.convert_to_tensor(X_test),
                EPSILON_LINF,
                np.inf,
                clip_min=0.0,
                clip_max=1.0
            ).numpy()

            del mlp_bin, logits_bin
            gc.collect()

        if RUN_MULTICLASS:
            mlp_multi = build_and_train_mlp(
                X_train,
                to_categorical(y_train_multi, len(class_names_multi)),
                len(class_names_multi)
            )

            logits_multi = Model(
                inputs=mlp_multi.input,
                outputs=mlp_multi.get_layer('logits').output
            )

            attacks_dict_multi['FGSM'] = fast_gradient_method(
                logits_multi,
                tf.convert_to_tensor(X_test),
                EPSILON_LINF,
                np.inf,
                clip_min=0.0,
                clip_max=1.0
            ).numpy()

            del mlp_multi, logits_multi
            gc.collect()

    if 'RANDOM_LINF' in ATTACKS_TO_RUN:
        print(f">>> A gerar Ruído Aleatório L_infinito (Eps={EPSILON_LINF})...")

        noise = np.random.uniform(
            -EPSILON_LINF,
            EPSILON_LINF,
            X_test.shape
        ).astype('float32')

        if RUN_BINARY:
            attacks_dict_bin['RANDOM_LINF'] = np.clip(X_test + noise, 0.0, 1.0)

        if RUN_MULTICLASS:
            attacks_dict_multi['RANDOM_LINF'] = np.clip(X_test + noise, 0.0, 1.0)

        del noise
        gc.collect()

    if 'RANDOM_L2' in ATTACKS_TO_RUN:
        print(f">>> A gerar Ruído Aleatório L_2 (Eps={EPSILON_L2})...")

        noise = np.random.normal(0, 1, X_test.shape).astype('float32')
        norms = np.linalg.norm(noise, axis=1, keepdims=True)
        norms[norms == 0] = 1e-10
        noise = noise * (EPSILON_L2 / norms)

        if RUN_BINARY:
            attacks_dict_bin['RANDOM_L2'] = np.clip(X_test + noise, 0.0, 1.0)

        if RUN_MULTICLASS:
            attacks_dict_multi['RANDOM_L2'] = np.clip(X_test + noise, 0.0, 1.0)

        del noise
        gc.collect()

    # 2.1 ATAQUE C&W
    if 'C&W' in ATTACKS_TO_RUN:
        print(">>> A gerar Ataque C&W L2 via surrogate MLP...")

        # Parâmetros do C&W.
        # Para teste rápido:
        #   CW_BINARY_SEARCH_STEPS = 3
        #   CW_MAX_ITERATIONS = 100 ou 200
        # Para experimento final mais forte:
        #   CW_BINARY_SEARCH_STEPS = 5
        #   CW_MAX_ITERATIONS = 500 ou 1000
        CW_BATCH_SIZE = 128
        CW_BINARY_SEARCH_STEPS = 5
        CW_MAX_ITERATIONS = 500
        CW_CONFIDENCE = 0.0
        CW_LEARNING_RATE = 5e-3
        CW_INITIAL_CONST = 1e-2
        CW_ABORT_EARLY = True

        def generate_cw_in_batches(
            logits_model,
            X_np,
            attack_name="C&W",
            batch_size=CW_BATCH_SIZE,
            binary_search_steps=CW_BINARY_SEARCH_STEPS,
            max_iterations=CW_MAX_ITERATIONS,
            confidence=CW_CONFIDENCE,
            learning_rate=CW_LEARNING_RATE,
            initial_const=CW_INITIAL_CONST,
            abort_early=CW_ABORT_EARLY,
        ):
            """
            Gera C&W L2 em lotes, sem duplicar a matriz adversarial em memória.

            Importante:
            - Não passamos y para o CleverHans, porque nesta implementação TF2
            y != None faz o ataque ser tratado como targeted.
            - A taxa de sucesso aqui é medida contra o surrogate:
            pred_surrogate(x_adv) != pred_surrogate(x_clean).
            """

            X_np = np.asarray(X_np, dtype=np.float32)
            n_samples = len(X_np)

            adv_x = np.empty_like(X_np, dtype=np.float32)
            l2_norms = np.zeros(n_samples, dtype=np.float32)
            success_flags = np.zeros(n_samples, dtype=bool)

            total_batches = (n_samples + batch_size - 1) // batch_size

            for i in tqdm(range(total_batches), desc=f"Gerando {attack_name}", unit="lote"):
                start_idx = i * batch_size
                end_idx = min((i + 1) * batch_size, n_samples)

                x_batch_np = X_np[start_idx:end_idx].astype(np.float32, copy=False)
                x_batch_tf = tf.convert_to_tensor(x_batch_np, dtype=tf.float32)
                x_batch_tf = tf.clip_by_value(x_batch_tf, 0.0, 1.0)

                # Predição limpa do surrogate
                clean_logits = logits_model(x_batch_tf, training=False)
                pred_clean = tf.argmax(clean_logits, axis=1).numpy()

                adv_batch = carlini_wagner_l2(
                    logits_model,
                    x_batch_tf,
                    batch_size=x_batch_tf.shape[0],
                    clip_min=0.0,
                    clip_max=1.0,
                    binary_search_steps=binary_search_steps,
                    max_iterations=max_iterations,
                    abort_early=abort_early,
                    confidence=confidence,
                    initial_const=initial_const,
                    learning_rate=learning_rate,
                )

                adv_batch_np = np.asarray(adv_batch, dtype=np.float32)
                adv_batch_np = np.clip(adv_batch_np, 0.0, 1.0)

                # Predição adversarial do surrogate
                adv_logits = logits_model(
                    tf.convert_to_tensor(adv_batch_np, dtype=tf.float32),
                    training=False
                )
                pred_adv = tf.argmax(adv_logits, axis=1).numpy()

                delta = adv_batch_np - x_batch_np
                l2_batch = np.linalg.norm(delta.reshape(delta.shape[0], -1), axis=1)

                adv_x[start_idx:end_idx] = adv_batch_np
                l2_norms[start_idx:end_idx] = l2_batch
                success_flags[start_idx:end_idx] = (pred_adv != pred_clean)

                del x_batch_np, x_batch_tf, adv_batch, adv_batch_np
                del clean_logits, adv_logits, pred_clean, pred_adv, delta, l2_batch
                gc.collect()

            stats = {
                "surrogate_success_rate": float(np.mean(success_flags)),
                "l2_mean": float(np.mean(l2_norms)),
                "l2_median": float(np.median(l2_norms)),
                "l2_max": float(np.max(l2_norms)),
                "l2_min": float(np.min(l2_norms)),
                "batch_size": int(batch_size),
                "binary_search_steps": int(binary_search_steps),
                "max_iterations": int(max_iterations),
                "confidence": float(confidence),
                "learning_rate": float(learning_rate),
                "initial_const": float(initial_const),
                "abort_early": bool(abort_early),
            }

            print(
                f"   [{attack_name}] Sucesso no surrogate: "
                f"{100 * stats['surrogate_success_rate']:.2f}% | "
                f"L2 médio={stats['l2_mean']:.4f} | "
                f"L2 mediano={stats['l2_median']:.4f} | "
                f"L2 máx={stats['l2_max']:.4f}"
            )

            return adv_x, stats

        cw_stats = {}

        if RUN_BINARY:
            print("   -> Treinando surrogate MLP para C&W (Binário)...")

            mlp_bin = build_and_train_mlp(
                X_train,
                to_categorical(y_train_bin, 2),
                2
            )

            logits_bin = Model(
                inputs=mlp_bin.input,
                outputs=mlp_bin.get_layer('logits').output
            )

            attacks_dict_bin['C&W'], cw_stats['binary'] = generate_cw_in_batches(
                logits_bin,
                X_test,
                attack_name="C&W Binário"
            )

            del mlp_bin, logits_bin
            gc.collect()
            tf.keras.backend.clear_session()

        if RUN_MULTICLASS:
            print("   -> Treinando surrogate MLP para C&W (Multiclasse)...")

            mlp_multi = build_and_train_mlp(
                X_train,
                to_categorical(y_train_multi, len(class_names_multi)),
                len(class_names_multi)
            )

            logits_multi = Model(
                inputs=mlp_multi.input,
                outputs=mlp_multi.get_layer('logits').output
            )

            attacks_dict_multi['C&W'], cw_stats['multiclass'] = generate_cw_in_batches(
                logits_multi,
                X_test,
                attack_name="C&W Multiclasse"
            )

            del mlp_multi, logits_multi
            gc.collect()
            tf.keras.backend.clear_session()

        # Salva estatísticas da geração C&W
        try:
            cw_stats_path = REPORTS_DIR / f"cw_generation_stats_{dataset_name}.csv"
            cw_rows = []

            for mode_name, stats in cw_stats.items():
                row_stats = {
                    "Dataset": dataset_name,
                    "Mode": mode_name,
                    "Attack": "C&W",
                }
                row_stats.update(stats)
                cw_rows.append(row_stats)

            if cw_rows:
                cw_stats_df = pd.DataFrame(cw_rows)
                file_exists = cw_stats_path.exists()
                cw_stats_df.to_csv(
                    cw_stats_path,
                    mode="a",
                    header=not file_exists,
                    index=False
                )

        except Exception as e:
            print(f"   [WARN] Não consegui salvar estatísticas do C&W: {repr(e)}")

        gc.collect()

    gc.collect()

    # 3. HIPERPARÂMETROS DA STANDARD WISARD
    param_grid = {
        'resolution': [1, 2, 4, 8, 10, 16, 32],
        'addressSize': [5, 10, 15, 20]
    }

    # Para testes rápidos, use algo como:
    # param_grid = {
    #     'resolution': [1],
    #     'addressSize': [5]
    # }

    # Pré-cálculos para Gaussian e Distributive
    X_mean = X_train.mean(axis=0)
    X_std = X_train.std(axis=0)
    X_std[X_std == 0] = 1e-8

    # 4. LOOP DA FAST COUNTING STANDARD WISARD
    for enc_type in ENCODING_TYPES:
        for res in param_grid['resolution']:
            print(f"\n[{dataset_name} | {enc_type.upper()} | Res={res}] Binarização...")

            custom_thresh = None

            if enc_type == 'gaussian':
                skews = [norm.ppf((i + 1) / (res + 1)) for i in range(res)]
                custom_thresh = X_mean[:, None] + (X_std[:, None] * skews)

            elif enc_type == 'distributive':
                percentiles = np.linspace(0, 100, res + 2)[1:-1]
                custom_thresh = np.percentile(X_train, percentiles, axis=0).T

            X_train_bin = process_data_vectorized_sequential(
                X_train,
                res,
                enc_type,
                custom_thresh
            )

            X_test_bin = process_data_vectorized_sequential(
                X_test,
                res,
                enc_type,
                custom_thresh
            )

            bin_adv_dict = {}
            if RUN_BINARY:
                for atk_name, X_adv_matrix in attacks_dict_bin.items():
                    bin_adv_dict[atk_name] = process_data_vectorized_sequential(
                        X_adv_matrix,
                        res,
                        enc_type,
                        custom_thresh
                    )

            multi_adv_dict = {}
            if RUN_MULTICLASS:
                for atk_name, X_adv_matrix in attacks_dict_multi.items():
                    multi_adv_dict[atk_name] = process_data_vectorized_sequential(
                        X_adv_matrix,
                        res,
                        enc_type,
                        custom_thresh
                    )

            modes_to_run = []

            if RUN_BINARY:
                modes_to_run.append('binary')

            if RUN_MULTICLASS:
                modes_to_run.append('multiclass')

            for mode in modes_to_run:
                if mode == 'binary':
                    y_train_curr = y_train_bin
                    y_test_curr = y_test_bin
                    class_names_curr = class_names_bin
                    active_adv_dict = bin_adv_dict

                else:
                    y_train_curr = y_train_multi
                    y_test_curr = y_test_multi
                    class_names_curr = class_names_multi
                    active_adv_dict = multi_adv_dict

                num_classes = len(class_names_curr)
                csv_name = REPORTS_DIR / f'standart_wisard_{dataset_name}_{mode}_{enc_type}.csv'

                for addr in param_grid['addressSize']:
                    print(
                        f"     [Addr={addr} | Mode={mode}] "
                        f"Treino/Avaliação FastCounting WiSARD com bleaching exato..."
                    )

                    parallel_out = evaluate_standard_persistent_parallel(
                        addr=addr,
                        X_train_bin=X_train_bin,
                        y_train_curr=y_train_curr,
                        X_test_bin=X_test_bin,
                        active_adv_dict=active_adv_dict,
                        n_jobs=N_JOBS,
                    )

                    results = parallel_out['results']
                    results_sorted = sorted(results, key=lambda r: r['start'])

                    yp_clean_str = []
                    yp_adv_str_dict = {atk: [] for atk in active_adv_dict.keys()}
                    pid_adv_time = {atk: {} for atk in active_adv_dict.keys()}

                    for r_worker in results_sorted:
                        pid = r_worker['pid']
                        yp_clean_str.extend(r_worker['yp_clean'])

                        for atk in active_adv_dict.keys():
                            yp_adv_str_dict[atk].extend(r_worker['yp_adv'][atk])
                            pid_adv_time[atk][pid] = (
                                pid_adv_time[atk].get(pid, 0.0)
                                + r_worker['infer_time_adv'][atk]
                            )

                    assert len(yp_clean_str) == len(y_test_curr)

                    for atk in active_adv_dict.keys():
                        assert len(yp_adv_str_dict[atk]) == len(y_test_curr)

                    yp_clean = np.array([int(y) for y in yp_clean_str])
                    yp_clean_prob = to_categorical(yp_clean, num_classes)

                    m_clean = calculate_miss_rates(
                        y_test_curr,
                        yp_clean,
                        yp_clean_prob,
                        "Clean",
                        class_names_curr
                    )

                    train_time_real = max([r['train_time'] for r in results])

                    for atk_name, yp_adv_str in yp_adv_str_dict.items():
                        yp_adv = np.array([int(y) for y in yp_adv_str])
                        yp_adv_prob = to_categorical(yp_adv, num_classes)

                        infer_time_adv_real = (
                            max(pid_adv_time[atk_name].values())
                            if pid_adv_time[atk_name]
                            else 0.0
                        )

                        m_adv = calculate_miss_rates(
                            y_test_curr,
                            yp_adv,
                            yp_adv_prob,
                            "Adv",
                            class_names_curr
                        )

                        acc_drop = m_clean['Clean_Acc'] - m_adv['Adv_Acc']

                        row = {
                            'Dataset': dataset_name,
                            'Attack': atk_name,
                            'Resolution': res,
                            'AddressSize': addr,
                            'TrainTime_s': train_time_real,
                            'InferTime_Adv_s': infer_time_adv_real,
                            'Acc_Drop_pp': acc_drop * 100
                        }

                        row.update(m_clean)
                        row.update(m_adv)

                        row_df = pd.DataFrame([row])
                        append_row_to_csv_safely(row_df, csv_name)

                        file_tag = (
                            f"fast_counting_wisard_{dataset_name}_{mode}_{enc_type}_"
                            f"{atk_name}_R{res}_A{addr}"
                        )

                        curve_path = CURVES_DIR / f"{file_tag}.npz"

                        save_curve_npz(
                            curve_path,
                            model_name=f"FastCounting WiSARD + Exact Bleaching ({enc_type})",
                            attack_name=atk_name,
                            y_true=y_test_curr,
                            y_prob_clean=yp_clean_prob,
                            y_prob_adv=yp_adv_prob,
                            class_names=class_names_curr
                        )

                    del (
                        parallel_out,
                        results,
                        results_sorted,
                        yp_clean_str,
                        yp_adv_str_dict,
                        yp_clean,
                        yp_clean_prob,
                        yp_adv,
                        yp_adv_prob
                    )
                    gc.collect()

            del X_train_bin, X_test_bin, bin_adv_dict, multi_adv_dict
            gc.collect()

    del attacks_dict_bin, attacks_dict_multi, X_train, X_test
    gc.collect()

print("\n🚀 EXPERIMENTO FAST COUNTING STANDARD WISARD CONCLUÍDO COM SUCESSO!")
print(f"CSV salvo em: {REPORTS_DIR}")
print(f"Curvas salvas em: {CURVES_DIR}")


>>> A INICIAR EXPERIMENTOS: BOT-IOT


C:\Users\root.REDE-LAGESED\AppData\Local\Temp\ipykernel_20988\53260640.py:47: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = df_train.select_dtypes(include=['object']).columns


>>> A aplicar Scaling e One-Hot Encoding...
>>> A gerar Ataque C&W L2 via surrogate MLP...
   -> Treinando surrogate MLP para C&W (Binário)...


Gerando C&W Binário: 100%|██████████| 860/860 [28:59<00:00,  2.02s/lote]


   [C&W Binário] Sucesso no surrogate: 0.00% | L2 médio=0.0000 | L2 mediano=0.0000 | L2 máx=0.1931

   -> Treinando surrogate MLP para C&W (Multiclasse)...


...

     [Addr=5 | Mode=multiclass] Treino/Avaliação FastCounting WiSARD com bleaching exato...
     [DISK] Memmap root: D:\wisard_memmap | livre≈9281.6 GB | estimado necessário≈82.7 GB
     [MEM] RAM disponível≈237.0 GB | RAM utilizável≈94.2 GB | pico estimado/worker≈4.1 GB | n_jobs seguro=23/23
     [Addr=10 | Mode=multiclass] Treino/Avaliação FastCounting WiSARD com bleaching exato...
     [DISK] Memmap root: D:\wisard_memmap | livre≈9281.6 GB | estimado necessário≈82.7 GB
     [MEM] RAM disponível≈237.0 GB | RAM utilizável≈94.2 GB | pico estimado/worker≈4.2 GB | n_jobs seguro=22/23
     [Addr=15 | Mode=multiclass] Treino/Avaliação FastCounting WiSARD com bleaching exato...
     [DISK] Memmap root: D:\wisard_memmap | livre≈9281.6 GB | estimado necessário≈82.7 GB
     [MEM] RAM disponível≈237.0 GB | RAM utilizável≈94.2 GB | pico estimado/worker≈5.4 GB | n_jobs seguro=17/23
     [Addr=20 | Mode=multiclass] Treino/Avaliação FastCounting WiSARD com bleaching exato...
     [DISK] Memmap root

Gerando C&W Binário: 100%|██████████| 3939/3939 [2:03:53<00:00,  1.89s/lote]  


   [C&W Binário] Sucesso no surrogate: 0.00% | L2 médio=0.0000 | L2 mediano=0.0000 | L2 máx=0.1537
   -> Treinando surrogate MLP para C&W (Multiclasse)...


Gerando C&W Multiclasse: 100%|██████████| 3939/3939 [2:05:45<00:00,  1.92s/lote]  


   [C&W Multiclasse] Sucesso no surrogate: 0.00% | L2 médio=0.0000 | L2 mediano=0.0000 | L2 máx=0.1537

[CICIDS | LINEAR | Res=1] Binarização...
     [Addr=5 | Mode=binary] Treino/Avaliação FastCounting WiSARD com bleaching exato...
     [DISK] Memmap root: D:\wisard_memmap | livre≈9281.6 GB | estimado necessário≈80.2 GB
     [MEM] RAM disponível≈238.6 GB | RAM utilizável≈95.2 GB | pico estimado/worker≈4.0 GB | n_jobs seguro=23/23
     [Addr=10 | Mode=binary] Treino/Avaliação FastCounting WiSARD com bleaching exato...
     [DISK] Memmap root: D:\wisard_memmap | livre≈9281.6 GB | estimado necessário≈80.2 GB
     [MEM] RAM disponível≈238.6 GB | RAM utilizável≈95.1 GB | pico estimado/worker≈4.0 GB | n_jobs seguro=23/23
     [Addr=15 | Mode=binary] Treino/Avaliação FastCounting WiSARD com bleaching exato...
     [DISK] Memmap root: D:\wisard_memmap | livre≈9281.6 GB | estimado necessário≈80.2 GB
     [MEM] RAM disponível≈238.5 GB | RAM utilizável≈95.1 GB | pico estimado/worker≈4.0 GB | n_j